# Lithospheric flexure with the gFlex component

This tutorial demonstrates the `landlab.components.gFlex` component, which wraps
the [gFlex](https://github.com/awickert/gFlex) lithospheric flexure solver
(Wickert, 2016).

**What this tutorial covers:**

* grid fields: what to write in, what to read out
* constructor parameters: elastic properties, infill density, boundary conditions
* load conversion: turning ice thickness, water depth, or sediment into stress [Pa]
* applying the deflection increment to topography
* updating elastic thickness (`T_e`) between timesteps via a grid field
* a complete time-stepping glacial isostasy example

**Requires:** `gflex >= 2.0.0` (install with `pip install gflex`).

## Grid fields

The component communicates through three CSDMS Standard Name node fields:

| Field | Direction | Units | Notes |
|---|---|---|---|
| `load__normal_component_of_stress` | input, **required** | Pa | Surface-normal stress from all load sources combined |
| `lithosphere__elastic_thickness` | input, *optional* | m | Re-read every `run_one_step()` call if present |
| `lithosphere__vertical_displacement` | output | m | Total deflection; downward is negative |

The required input field must be created **before** constructing the component.
The optional field and the output field are created automatically.

In [ ]:
import numpy as np

from landlab import RasterModelGrid
from landlab.components import gFlex

## Basic usage

Create a grid, add the required load field, construct the component, run.

In [ ]:
# 40 x 40 grid, 10 km cell spacing -> 400 km domain
mg = RasterModelGrid((40, 40), xy_spacing=10_000.0)

# Required: create the load field before constructing the component
mg.add_zeros("load__normal_component_of_stress", at="node")

# Apply a square load in the centre: 500 m of ice (rho_ice * g * h)
rho_ice = 910.0  # kg m^-3
g = 9.81  # m s^-2
load_view = mg.at_node["load__normal_component_of_stress"].reshape(mg.shape)
load_view[15:25, 15:25] = rho_ice * g * 500.0

# Construct the component with default parameters
gf = gFlex(
    mg,
    elastic_thickness=35_000.0,  # m — 35 km
)

# Run
gf.run_one_step()

w = mg.at_node["lithosphere__vertical_displacement"]
print(f"Max subsidence: {w.min():.2f} m")  # most negative value = deepest point

## Constructor parameters

| Parameter | Default | Description |
|---|---|---|
| `Youngs_modulus` | `65e9` | Young's modulus *E* [Pa] |
| `Poissons_ratio` | `0.25` | Poisson's ratio *ν* |
| `rho_mantle` | `3300.0` | Mantle density *ρ_m* [kg m⁻³] |
| `rho_fill` | `0.0` | Infill density *ρ_fill* [kg m⁻³]; see below |
| `elastic_thickness` | `35000.0` | Elastic thickness *T_e* [m]; scalar, field name, or array |
| `method` | `'fd'` | Solver: `'fd'` (finite difference), `'fft'`, `'sas'`, `'sas_ng'` |
| `bc_west/east/north/south` | `'no_outside_loads'` | Boundary condition for each edge |
| `g` | `scipy.constants.g` | Gravitational acceleration [m s⁻²] |

All grid spacings and lengths must be in **metres**.

## Load conversion

gFlex is material-agnostic. The `load__normal_component_of_stress` field [Pa]
must contain the surface-normal compressive stress from *all* load sources.
The caller converts source-specific quantities to Pa:

```python
# Glacial ice
mg.at_node["load__normal_component_of_stress"][:] = rho_ice * g * ice_thickness

# Sediment or volcanic load
mg.at_node["load__normal_component_of_stress"][:] = rho_sed * g * sed_thickness

# Multiple sources: sum them
mg.at_node["load__normal_component_of_stress"][:] = (
    rho_ice * g * ice_thickness + rho_water * g * water_depth
)
```

Positive values are compressive (pushing down, causing subsidence).

## Infill density (`rho_fill`)

`rho_fill` is the density of the material that **fills the flexural depression**,
not the material applying the load. It provides an upward restoring force in
addition to mantle buoyancy.

| Setting | `rho_fill` [kg m⁻³] | When to use |
|---|---|---|
| Air | `0.0` *(default)* | Subaerial basin exposed to atmosphere |
| Seawater | `1030` | Submarine basin or marine ice sheet |
| Fresh water | `1000` | Proglacial or subglacial lake |
| Sediment | `2000`–`2700` | Sediment-filled foreland basin |

`rho_fill` is set at construction time. If the infill material changes during a
simulation (e.g. a subaerial basin floods), reconstruct the component.

## Boundary conditions

The default `'no_outside_loads'` (alias `'infinite'`) models an infinite plate
with no loads outside the domain — the safest choice for most applications.

Other options (see the
[gFlex boundary conditions docs](https://gflex.readthedocs.io/en/latest/boundary_conditions.html)
for diagrams and physical interpretation):

| Canonical name | Alias | Meaning |
|---|---|---|
| `'no_outside_loads'` | `'infinite'` | Infinite plate (default) |
| `'zero_displacement_zero_slope'` | `'clamped'` | Clamped edge |
| `'zero_displacement_zero_moment'` | `'pinned'` | Pinned edge |
| `'zero_moment_zero_shear'` | `'free'` | Free broken-plate end |
| `'zero_slope_zero_shear'` | `'mirror'` | Symmetric (mirror) edge |
| `'periodic'` | — | Periodic; must be paired west–east or north–south |

Any string in `gflex.VALID_BC_STRINGS_2D` is accepted. A `dict` of prescribed
values may be passed for inhomogeneous (nested-domain) boundary conditions.

## Applying deflection to topography

`lithosphere__vertical_displacement` holds the **total** instantaneous deflection,
not the increment since the last call. In a time-stepping loop, apply only the
*change* to topography:

In [ ]:
mg2 = RasterModelGrid((40, 40), xy_spacing=10_000.0)
mg2.add_zeros("load__normal_component_of_stress", at="node")
mg2.add_zeros("topographic__elevation", at="node")

gf2 = gFlex(mg2, elastic_thickness=35_000.0)

w_prev = np.zeros(mg2.number_of_nodes)

for step in range(3):
    # Load grows each step
    load = mg2.at_node["load__normal_component_of_stress"].reshape(mg2.shape)
    load[:] = 0.0
    load[15:25, 15:25] = rho_ice * g * 500.0 * (step + 1)

    gf2.run_one_step()

    w = mg2.at_node["lithosphere__vertical_displacement"]
    mg2.at_node["topographic__elevation"] += w - w_prev
    w_prev = w.copy()

print(
    f"Final max subsidence in topo: {mg2.at_node['topographic__elevation'].min():.2f} m"
)

## Variable elastic thickness

There are three ways to set *T_e* at construction:

```python
# 1. Scalar — constant, LU factorisation cached automatically
gf = gFlex(mg, elastic_thickness=35_000.0)

# 2. Array — spatially variable, fixed for the run
Te = np.linspace(15e3, 35e3, mg.number_of_nodes)
gf = gFlex(mg, elastic_thickness=Te)

# 3. Name of an existing node field — reads T_e from the field at construction
mg.add_field("lithosphere__elastic_thickness", Te, at="node")
gf = gFlex(mg, elastic_thickness="lithosphere__elastic_thickness")
```

To **update *T_e* between timesteps**, create the `lithosphere__elastic_thickness`
field and write to it before each `run_one_step()` call. The component reads
this field on every call if it exists, enabling BMI-style updates:

In [ ]:
mg3 = RasterModelGrid((40, 40), xy_spacing=10_000.0)
mg3.add_zeros("load__normal_component_of_stress", at="node")

# Load field for T_e — created before the component so run_one_step() will read it
mg3.add_zeros("lithosphere__elastic_thickness", at="node")
mg3.at_node["lithosphere__elastic_thickness"][:] = 35_000.0

mg3.at_node["load__normal_component_of_stress"].reshape(mg3.shape)[15:25, 15:25] = (
    rho_ice * g * 500.0
)

gf3 = gFlex(mg3, elastic_thickness=35_000.0)
gf3.run_one_step()
w_thin = mg3.at_node["lithosphere__vertical_displacement"].min()

# Double T_e via the field — next call uses the new value
mg3.at_node["lithosphere__elastic_thickness"][:] = 70_000.0
gf3.run_one_step()
w_thick = mg3.at_node["lithosphere__vertical_displacement"].min()

print(f"T_e = 35 km → max subsidence {w_thin:.2f} m")
print(f"T_e = 70 km → max subsidence {w_thick:.2f} m  (stiffer plate, less deflection)")

## Complete example: glacial isostasy

An ice sheet grows over 20 timesteps on a flat plate with seawater infill
(`rho_fill = 1030`). Topography is updated each step.

In [ ]:
import matplotlib.pyplot as plt

nrows = ncols = 60
dx = 10_000.0  # 10 km
rho_ice = 910.0
rho_sw = 1030.0
g = 9.81

mg4 = RasterModelGrid((nrows, ncols), xy_spacing=dx)
mg4.add_zeros("topographic__elevation", at="node")
mg4.add_zeros("load__normal_component_of_stress", at="node")

gf4 = gFlex(
    mg4,
    elastic_thickness=35_000.0,
    rho_fill=rho_sw,
)

n_steps = 20
w_prev = np.zeros(mg4.number_of_nodes)
cx = cy = 0.5 * nrows * dx
XX, YY = np.meshgrid(np.arange(ncols) * dx, np.arange(nrows) * dx)

for step in range(n_steps):
    r = (step + 1) / n_steps
    # Gaussian ice sheet growing in radius and height
    sigma = r * 150e3
    h_ice = 2000.0 * r * np.exp(-((XX - cx) ** 2 + (YY - cy) ** 2) / (2 * sigma**2))
    mg4.at_node["load__normal_component_of_stress"][:] = rho_ice * g * h_ice.ravel()

    gf4.run_one_step()

    w = mg4.at_node["lithosphere__vertical_displacement"]
    mg4.at_node["topographic__elevation"] += w - w_prev
    w_prev = w.copy()

# Plot final deflection
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im0 = axes[0].imshow(
    h_ice, origin="upper", extent=[0, ncols * dx / 1e3, nrows * dx / 1e3, 0]
)
axes[0].set_title("Ice thickness [m]")
axes[0].set_xlabel("x [km]")
axes[0].set_ylabel("y [km]")
plt.colorbar(im0, ax=axes[0])

w_grid = mg4.at_node["lithosphere__vertical_displacement"].reshape(mg4.shape)
wmax = np.abs(w_grid).max()
im1 = axes[1].imshow(
    w_grid,
    origin="upper",
    extent=[0, ncols * dx / 1e3, nrows * dx / 1e3, 0],
    cmap="RdBu",
    vmin=-wmax,
    vmax=wmax,
)
axes[1].set_title("Flexural deflection [m]\n(blue = subsidence, red = uplift)")
axes[1].set_xlabel("x [km]")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

print(f"Maximum subsidence: {w_grid.min():.1f} m")
print(f"Forebulge height:   {w_grid.max():.2f} m")

## Relationship to the gFlex BMI

The Landlab component and the gFlex CSDMS BMI (`gflex.BmiGflex`) wrap the same
underlying solver and use the same CSDMS Standard Name fields. The differences
are framework conventions:

| | gFlex BMI | Landlab component |
|---|---|---|
| Lifecycle | `initialize` / `update` / `finalize` | `__init__` / `run_one_step` |
| Load input | `set_value("load__normal_component_of_stress", q)` | `grid.at_node["load__normal_component_of_stress"][:] = q` |
| *T_e* update | `set_value("lithosphere__elastic_thickness", te)` | write to `grid.at_node["lithosphere__elastic_thickness"]` |
| Deflection output | `get_value("lithosphere__vertical_displacement", w)` | `grid.at_node["lithosphere__vertical_displacement"]` |
| Physical constants | Grid-1 scalar BMI variables | Constructor keyword arguments |

## References

Wickert, A. D. (2016). Open-source modular solutions for flexural isostasy:
gFlex v1.0. *Geoscientific Model Development*, 9(3), 997–1017.
https://doi.org/10.5194/gmd-9-997-2016